In [ ]:
from pathlib import Path
import re
import pandas as pd

from sklearn.feature_extraction.text import TfidfVectorizer

BASE_DIR = Path.cwd().parent
DATA_PATH = BASE_DIR / "data" / "customer_support_tickets.csv"

df = pd.read_csv(DATA_PATH)

df.head()

In [ ]:
def clean_text(text):
    text = str(text).lower()
    text = re.sub(r"http\S+|www\S+", " ", text)
    text = re.sub(r"[^a-z\s]", " ", text)
    text = re.sub(r"\s+", " ", text)
    return text.strip()

In [ ]:
df["text"] = (
    df["Ticket Subject"].fillna("")
    + " "
    + df["Ticket Description"].fillna("")
)

df["clean_text"] = df["text"].apply(
    clean_text
)

df[
    ["text", "clean_text"]
].head(10)

In [ ]:
df = df.dropna(
    subset=[
        "clean_text",
        "Ticket Type",
        "Ticket Priority",
        "Resolution"
    ]
).copy()

df.shape

In [ ]:
priority_mapping = {
    "low": "low",
    "medium": "medium",
    "high": "high",
    "critical": "high"
}

df["urgency"] = (
    df["Ticket Priority"]
    .astype(str)
    .str.lower()
    .map(priority_mapping)
)

df["urgency"].value_counts()

In [ ]:
vectorizer = TfidfVectorizer(
    max_features=10000,
    ngram_range=(1, 2),
    min_df=2,
    sublinear_tf=True
)

X_tfidf = vectorizer.fit_transform(
    df["clean_text"]
)

X_tfidf.shape

In [ ]:
feature_names = vectorizer.get_feature_names_out()

feature_names[:50]

In [ ]:
tfidf_values = X_tfidf.mean(axis=0).A1

tfidf_features = pd.DataFrame({
    "feature": feature_names,
    "importance": tfidf_values
}).sort_values(
    "importance",
    ascending=False
)

tfidf_features.head(20)